In [1]:
import pandas as pd
from ydata_profiling import ProfileReport
import datamart_profiler
import openai
from dotenv import load_dotenv
from openai import OpenAI
import os
import glob
import json
import sys
sys.path.append(r'c:\Users\ricca\Documents\Polimi\tesi\code')
import utils
import warnings
warnings.filterwarnings('ignore')

from tqdm import tqdm  # For progress bar
from multiprocessing import Pool

import importlib
importlib.reload(utils)


<module 'utils' from 'c:\\Users\\ricca\\Documents\\Polimi\\tesi\\code\\utils.py'>

In [2]:
csv_files = glob.glob('benchmark/datasets_90_csv/*.csv')
print(f"Found {len(csv_files)} csv files")

Found 49 csv files


We generate the data profiles for the benchmark datasets

In [3]:
data_profiles = []
for file_path in tqdm(csv_files, desc="Generating Data Profiles"):
    try:
        profile = utils.generate_data_profile(file_path)
        data_profiles.append(profile)
    except Exception as e:
        print(f"Error processing {file_path}: {str(e)}")

print(f"\nProcessed {len(data_profiles)} files successfully")

Generating Data Profiles: 100%|██████████| 49/49 [12:07<00:00, 14.84s/it]


Processed 49 files successfully


We generate the semantic profiles for the diabetes datasets

In [4]:
semantic_profiles = []

for data_profile in data_profiles:
    prompt = utils.generate_prompt(data_profile, utils.TEMPLATE_SEMANTIC, utils.RESPONSE_EXAMPLE_SEMANTIC)
    response = utils.call_openai_api(prompt, "o3-mini")
    semantic_profiles.append(response.choices[0].message.content)

In [5]:
dataset_info = [
    {
        "dataset_name": title,
        "data_profile": data_profiles[i],
        "semantic_profile": semantic_profiles[i],
        "instructions": None
    }
    for i, title in enumerate(csv_files)
]

In [6]:
for dataset in dataset_info:
    prompt = utils.generate_prompt_instructions(dataset.get("dataset_name"), dataset.get("data_profile"), dataset.get("semantic_profile"))
    dataset["instructions"] = utils.json_to_dict(utils.call_openai_api(prompt, "o1-mini").choices[0].message.content)


In [7]:
dataset_info[6].get("instructions")

{'queries': [{'query': 'Find a dataset containing statistics on nonmarital births over multiple years.'},
  {'query': 'Search for datasets with demographic information related to birth rates.'},
  {'query': 'Show me data on the percentage of nonmarital births from 1970 to 2015.'},
  {'query': 'Locate a dataset that categorizes nonmarital birth percentages by different age groups.'},
  {'query': 'Find datasets that include the mean, maximum, and minimum percentages of nonmarital births.'},
  {'query': 'Search for time-series data on nonmarital birth rates across various age demographics.'},
  {'query': 'Show datasets that aggregate birth statistics by year and age group.'},
  {'query': 'Find data with a numeric column representing the year and its related birth statistics.'},
  {'query': 'Locate a dataset that provides quantitative measurements of nonmarital birth percentages.'},
  {'query': 'Search for demographic datasets that include both temporal resolution by year and age group cla

In [11]:
import pickle

with open("dataset_info_benchmark.pkl", "wb") as f:
    pickle.dump(dataset_info, f)